# EXP-014 — Model-package OOF audit

## tl;dr

The package contains valid group-OOF predictions for 3,783,989 rows and 773 wells. Its postprocessed blend scores **10.6702 RMSE**, versus **15.9099** for last-known carry-forward, and improves 75.3% of wells. It is not a safe replacement for the current anchor: 24.7% of wells worsen and the worst well reaches 49.52 RMSE. A package/anchor blend cannot be selected honestly until anchor OOF predictions are exported on the same IDs.

## Context & Methods

Decision: determine whether `pilkwang/rogii-model-package` justifies another code submission.

### Key assumptions

- `train_gt.parquet` and `*_oof.npy` retain identical row order.
- OOF folds are grouped by well as documented in the package reports.
- Last-known carry-forward is a sanity baseline, not our production anchor.
- No package/anchor weight is recommended without aligned anchor OOF.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("data")
truth = pd.read_parquet(DATA / "train_gt.parquet")
predictions = {p.stem: np.load(p) for p in DATA.glob("*_oof*.npy")}
rmse = lambda y, p: float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))
y = truth["target_delta_from_last_known"].to_numpy(dtype=float)

## Data

In [2]:
quality = {
    "rows": len(truth),
    "wells": int(truth["well_id"].nunique()),
    "unique_ids": bool(truth["id"].is_unique),
    "finite_target_rate": float(np.isfinite(y).mean()),
    "prediction_lengths_match": all(len(p) == len(truth) for p in predictions.values()),
    "finite_prediction_rate_min": min(float(np.isfinite(p).mean()) for p in predictions.values()),
}
quality

{'rows': 3783989,
 'wells': 773,
 'unique_ids': True,
 'finite_target_rate': 1.0,
 'prediction_lengths_match': True,
 'finite_prediction_rate_min': 1.0}

## Results

In [3]:
rows = [{"candidate": "last_known", "rmse": rmse(y, np.zeros_like(y))}]
for name, pred in sorted(predictions.items()):
    rows.append({"candidate": name, "rmse": rmse(y, pred)})
metrics = pd.DataFrame(rows).sort_values("rmse")
metrics

,candidate,rmse
2,blend_oof_postprocessed,10.670211
1,blend_oof,10.710645
3,catboost_oof,11.024700
5,lgb_oof,11.247567
4,hgb_oof,11.255841
6,sequence_tcn_oof,11.257960
7,xgb_oof,11.361523
0,last_known,15.909853


In [4]:
package = predictions["blend_oof_postprocessed"]
shrink = float(np.dot(package, y) / np.dot(package, package))
X = np.column_stack([np.ones(len(package)), package])
intercept, slope = np.linalg.lstsq(X, y, rcond=None)[0]
calibration = pd.DataFrame([
    {"candidate": "package", "intercept": 0.0, "slope": 1.0, "rmse": rmse(y, package)},
    {"candidate": "shrink_only", "intercept": 0.0, "slope": shrink, "rmse": rmse(y, shrink * package)},
    {"candidate": "affine_in_sample_diagnostic", "intercept": intercept, "slope": slope, "rmse": rmse(y, intercept + slope * package)},
])
calibration

,candidate,intercept,slope,rmse
0,package,0.000000,1.000000,10.670211
1,shrink_only,0.000000,1.045497,10.657822
2,affine_in_sample_diagnostic,-0.232879,1.048678,10.655338


In [5]:
well_frame = pd.DataFrame({"well": truth["well_id"].astype(str), "target": y, "package": package})
well_metrics = well_frame.groupby("well", sort=False).apply(
    lambda g: pd.Series({
        "rows": len(g),
        "package_rmse": rmse(g["target"], g["package"]),
        "last_known_rmse": rmse(g["target"], np.zeros(len(g))),
    }),
    include_groups=False,
)
well_metrics["gain_vs_last_known"] = well_metrics["last_known_rmse"] - well_metrics["package_rmse"]
well_summary = pd.Series({
    "package_win_rate": float((well_metrics["gain_vs_last_known"] > 0).mean()),
    "median_gain": float(well_metrics["gain_vs_last_known"].median()),
    "gain_p10": float(well_metrics["gain_vs_last_known"].quantile(0.10)),
    "gain_p90": float(well_metrics["gain_vs_last_known"].quantile(0.90)),
    "package_well_rmse_median": float(well_metrics["package_rmse"].median()),
    "package_well_rmse_p90": float(well_metrics["package_rmse"].quantile(0.90)),
    "package_well_rmse_max": float(well_metrics["package_rmse"].max()),
})
well_summary

package_win_rate             0.752911
median_gain                  2.833367
gain_p10                    -2.882714
gain_p90                    12.705127
package_well_rmse_median     7.227768
package_well_rmse_p90       15.338803
package_well_rmse_max       49.523764
dtype: float64

In [6]:
well_metrics.sort_values("gain_vs_last_known").head(10)

,rows,package_rmse,last_known_rmse,gain_vs_last_known
well,,,,
896d15b9,4279.0,49.523764,29.784607,-19.739157
7224331b,6063.0,14.607041,4.338260,-10.268781
204cc64b,4805.0,16.350961,6.094880,-10.256081
dc7da28f,4774.0,19.562820,10.223244,-9.339575
353e5502,6062.0,14.473013,5.639361,-8.833652
660d9546,4732.0,18.054502,9.433758,-8.620744
9dfff011,4319.0,13.351208,4.845970,-8.505238
a76db406,4672.0,12.039610,3.704320,-8.335290
7850c72e,5497.0,20.640390,12.350660,-8.289730


## Takeaways

1. The package OOF evidence is structurally trustworthy: complete, finite, unique IDs, and grouped across 773 wells.
2. The package has substantial standalone signal, but a material failure tail; it should only be a bounded residual.
3. The package's best global shrink is about 1.045, so the current 1% gate is not supported by package-only OOF—it was chosen as a safety move, not an accuracy optimum.
4. The current Kaggle run moved zero rows because the 25 ft guard fired. Raising that threshold blindly is not justified.
5. Required next evidence: export the production anchor OOF on these same IDs, then optimize a bounded package residual with fold and worst-well guardrails.